# Model deployment

## Import usual packages

In [49]:
import torch
import torchvision
import torchinfo as ti

In [50]:
import torch.nn as nn

In [51]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [52]:
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights, vit_b_16, ViT_B_16_Weights

In [53]:
from torchvision import datasets, transforms

In [77]:
from torch.utils.data import DataLoader

In [54]:
from pathlib import Path
import os

In [55]:
from zipfile import ZipFile

In [56]:
from copy import deepcopy

## Constant

In [57]:
SIZE_INFO = (1,3,224,224)

## Creating an EfficientNet B2 model

In [58]:
eff_net_model = efficientnet_b2(weights=EfficientNet_B2_Weights.IMAGENET1K_V1)

In [59]:
ti.summary(eff_net_model, input_size=SIZE_INFO)

Layer (type:depth-idx)                                  Output Shape              Param #
EfficientNet                                            [1, 1000]                 --
├─Sequential: 1-1                                       [1, 1408, 7, 7]           --
│    └─Conv2dNormActivation: 2-1                        [1, 32, 112, 112]         --
│    │    └─Conv2d: 3-1                                 [1, 32, 112, 112]         864
│    │    └─BatchNorm2d: 3-2                            [1, 32, 112, 112]         64
│    │    └─SiLU: 3-3                                   [1, 32, 112, 112]         --
│    └─Sequential: 2-2                                  [1, 16, 112, 112]         --
│    │    └─MBConv: 3-4                                 [1, 16, 112, 112]         1,448
│    │    └─MBConv: 3-5                                 [1, 16, 112, 112]         612
│    └─Sequential: 2-3                                  [1, 24, 56, 56]           --
│    │    └─MBConv: 3-6                                

## Creating a ViT-Base model

In [60]:
vit_model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)

In [61]:
ti.summary(vit_model, input_size=SIZE_INFO)

Layer (type:depth-idx)                        Output Shape              Param #
VisionTransformer                             [1, 1000]                 768
├─Conv2d: 1-1                                 [1, 768, 14, 14]          590,592
├─Encoder: 1-2                                [1, 197, 768]             151,296
│    └─Dropout: 2-1                           [1, 197, 768]             --
│    └─Sequential: 2-2                        [1, 197, 768]             --
│    │    └─EncoderBlock: 3-1                 [1, 197, 768]             7,087,872
│    │    └─EncoderBlock: 3-2                 [1, 197, 768]             7,087,872
│    │    └─EncoderBlock: 3-3                 [1, 197, 768]             7,087,872
│    │    └─EncoderBlock: 3-4                 [1, 197, 768]             7,087,872
│    │    └─EncoderBlock: 3-5                 [1, 197, 768]             7,087,872
│    │    └─EncoderBlock: 3-6                 [1, 197, 768]             7,087,872
│    │    └─EncoderBlock: 3-7             

## Create the classificatgion head

In [62]:
eff_net_model.classifier

Sequential(
  (0): Dropout(p=0.3, inplace=True)
  (1): Linear(in_features=1408, out_features=1000, bias=True)
)

In [63]:
vit_model.heads

Sequential(
  (head): Linear(in_features=768, out_features=1000, bias=True)
)

In [64]:
def create_classification_head(model:str):
    if model.lower() != "vit":
        in_feature = 1408
        class_head = nn.Sequential(
            Dropout(p=0.3, inplace=True),
            Linear(in_features=in_feature, out_features=3, bias=True)
        )
        return class_head
    return nn.Sequential(Linear(in_features=768, out_features=3, bias=True))

## Downloading the data

In [65]:
!wget https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip

--2026-07-15 12:37:26--  https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/data/pizza_steak_sushi_20_percent.zip [following]
--2026-07-15 12:37:27--  https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/data/pizza_steak_sushi_20_percent.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8001::154, 2606:50c0:8003::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8001::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 31491084 (30M) [application/zip]
Saving to: ‘pizza_steak_sushi_20_percent.zip’

pizza_steak_sushi_2 100%[===================>]  30.03M  31.0MB/s  

In [66]:
current_folder = Path(".")
for file in current_folder.iterdir():
    if file.suffix == ".zip":
        with  ZipFile("pizza_steak_sushi_20_percent.zip", mode="r") as file:
            file.extractall("data/")
        os.remove("pizza_steak_sushi_20_percent.zip")
        print("Zip file deleted and extracted")

Zip file deleted and extracted


In [67]:
train_dir = Path(".")/ "data"/ "train"
test_dir = Path(".") / "data" / "test"

In [68]:
train_dir, test_dir

(PosixPath('data/train'), PosixPath('data/test'))

In [69]:
# Check if the folders are well located in the train and test directory
for folder in train_dir.iterdir():
    print(folder)

data/train/steak
data/train/sushi
data/train/pizza


## Create the transforms

In [70]:
EfficientNet_B2_Weights.IMAGENET1K_V1.transforms()

ImageClassification(
    crop_size=[288]
    resize_size=[288]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [71]:
ViT_B_16_Weights.IMAGENET1K_V1.transforms()

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [72]:
train_transform_eff_net = transforms.Compose([
    transforms.Resize(size=(288,288)),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform_eff_net = deepcopy(train_transform_eff_net)

In [73]:
train_transform_vit = transforms.Compose([
    transforms.Resize(size=(224,224)),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform_vit = deepcopy(train_transform_vit)

## Creating the dataset and dataloders

In [74]:
train_dataset_eff_net = datasets.ImageFolder(root=train_dir, transform=train_transform_eff_net)
test_dataset_eff_net = datasets.ImageFolder(root=test_dir, transform = test_transform_eff_net)

In [78]:
train_dataloader_eff_net = DataLoader(dataset=train_dataset_eff_net, 
                                      batch_size=32, 
                                      shuffle=True, 
                                      num_workers=os.cpu_count()-4, 
                                      prefetch_factor=2)
test_dataloader_eff_net = DataLoader(dataset=test_dataset_eff_net,
                                     batch_size=32,
                                     num_workers=os.cpu_count()-4,
                                     prefetch_factor=2)

In [75]:
train_dataset_vit = datasets.ImageFolder(root=train_dir, transform=train_transform_vit)
test_dataset_vit = datasets.ImageFolder(root=test_dir, transform=test_transform_vit)

In [79]:
train_dataloader_vit = DataLoader(dataset=train_dataset_vit, 
                                  batch_size=32,
                                  shuffle = True,
                                  prefetch_factor= 2,
                                  num_workers=os.cpu_count()-4)
test_dataloader_vit = DataLoader(dataset=test_dataset_vit,
                                 batch_size=32,
                                 prefetch_factor=2,
                                 num_workers=os.cpu_count()-4)

## Creating the training and testing loop

In [80]:
# Training loop
def train():
    pass

In [81]:
# Testing loop
def test():
    pass

In [82]:
# Engine loop
def train_and_evalutate():
    pass